# 01 · Merge de datos abiertos de Dengue (SINAVE / DGE)

**Proyecto final – Análisis Estadístico Multivariado**
Objetivo general: regresión logística binaria sobre `DEFUNCION` en casos confirmados de dengue.
Este notebook **solo** construye la base unificada; la limpieza y el análisis van en notebooks posteriores.

### ¿Cómo vienen los datos?
- Fuente: [Datos Abiertos – Bases Históricas de ETV](https://www.gob.mx/salud/documentos/datos-abiertos-bases-historicas-de-enfermedades-transmitidas-por-vector) (dic-2020 → nov-2026) y el repo [RodrigoZepeda/DengueMX](https://github.com/RodrigoZepeda/DengueMX) (dic-2020 → oct-2024).
- Cada `datos_abiertos_dengue_DDMMYY.zip` es una **foto acumulada del año en curso**: incluye todos los casos del año hasta esa fecha, no solo los de esa semana. Al inicio de enero, las fotos todavía traen los casos del año anterior.
- Entre una foto y otra, los registros **se corrigen**: cambia `ESTATUS_CASO`, `DEFUNCION`, etc.

### Estrategia
1. Leer las fotos en orden cronológico (según la fecha del nombre del archivo).
2. Ir acumulando y quedarse con la **versión más reciente de cada `ID_REGISTRO`**. Es el mismo criterio que usa `format_dataset.R` en el repo original.
3. Guardar metadatos de trazabilidad por registro: en cuántas fotos apareció, la primera y la última, si cambió `DEFUNCION` y si sigue en la foto final de su año.

In [1]:
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

DATA_DIR = Path("datos-abiertos")
OUT_DIR = Path("datos-procesados")
OUT_DIR.mkdir(exist_ok=True)

## 1. Inventario de archivos

In [2]:
zips = sorted(DATA_DIR.glob("datos_abiertos_dengue_*.zip"))

inventario = pd.DataFrame({
    "archivo": [z.name for z in zips],
    "fecha_archivo": [pd.to_datetime(re.search(r"(\d{6})\.zip$", z.name).group(1), format="%d%m%y") for z in zips],
    "ruta": zips,
}).sort_values("fecha_archivo").reset_index(drop=True)
inventario["anio_archivo"] = inventario["fecha_archivo"].dt.year

print(f"{len(inventario)} archivos: {inventario.fecha_archivo.min():%d-%m-%Y} → {inventario.fecha_archivo.max():%d-%m-%Y}")
inventario.groupby("anio_archivo").agg(n_archivos=("archivo", "size"),
                                       primero=("fecha_archivo", "min"),
                                       ultimo=("fecha_archivo", "max"))

294 archivos: 10-12-2020 → 05-11-2026


,n_archivos,primero,ultimo
anio_archivo,,,
2020,4,2020-12-10,2020-12-31
2021,50,2021-01-07,2021-12-30
2022,53,2022-01-06,2022-12-29
2023,51,2023-01-05,2023-12-27
2024,50,2024-01-03,2024-12-19
2025,48,2025-01-08,2025-12-26
2026,38,2026-01-08,2026-11-05


## 2. Lectura de cada foto
- Algunos zips traen basura de macOS (`__MACOSX/`) o el CSV con otro nombre. Por eso se elige el único `.csv` válido.
- Todo se lee como **texto** (`dtype=str`) para no perder información. Los tipos se ajustan al final.

In [3]:
def leer_foto(ruta: Path) -> pd.DataFrame:
    with zipfile.ZipFile(ruta) as z:
        csvs = [n for n in z.namelist() if n.lower().endswith(".csv") and not n.startswith("__MACOSX")]
        assert len(csvs) == 1, f"{ruta.name}: {csvs}"
        with z.open(csvs[0]) as fh:
            df = pd.read_csv(fh, dtype=str, encoding="utf-8")
    df.columns = df.columns.str.strip().str.lstrip("﻿")
    return df

# Validamos que todas las fotos tengan exactamente las mismas columnas
COLUMNAS = list(leer_foto(inventario.ruta.iloc[-1]).columns)
print(len(COLUMNAS), COLUMNAS)

28 ['FECHA_ACTUALIZACION', 'ID_REGISTRO', 'SEXO', 'EDAD_ANOS', 'ENTIDAD_RES', 'MUNICIPIO_RES', 'HABLA_LENGUA_INDIG', 'INDIGENA', 'ENTIDAD_UM_NOTIF', 'MUNICIPIO_UM_NOTIF', 'INSTITUCION_UM_NOTIF', 'FECHA_SIGN_SINTOMAS', 'TIPO_PACIENTE', 'HEMORRAGICOS', 'DIABETES', 'HIPERTENSION', 'ENFERMEDAD_ULC_PEPTICA', 'ENFERMEDAD_RENAL', 'INMUNOSUPR', 'CIRROSIS_HEPATICA', 'EMBARAZO', 'DEFUNCION', 'DICTAMEN', 'TOMA_MUESTRA', 'RESULTADO_PCR', 'ESTATUS_CASO', 'ENTIDAD_ASIG', 'MUNICIPIO_ASIG']


## 3. Merge incremental + deduplicación por `ID_REGISTRO`
Juntar las ~20 millones de filas de una sola vez satura la memoria. Por eso el merge se hace foto por foto:
`acumulado = concat(acumulado, foto).drop_duplicates(ID_REGISTRO, keep="last")`.
En paralelo se lleva una bitácora ligera (`ID`, número de foto, `DEFUNCION`) para la trazabilidad.

In [4]:
acumulado = None
bitacora = []        # (ID_REGISTRO, idx_foto, DEFUNCION) de cada aparición
filas_leidas = 0
dups_internos = 0

for idx, fila in inventario.iterrows():
    foto = leer_foto(fila.ruta)
    assert list(foto.columns) == COLUMNAS, f"Columnas distintas en {fila.archivo}"
    filas_leidas += len(foto)
    dups_internos += foto.ID_REGISTRO.duplicated().sum()

    foto["ARCHIVO_FUENTE"] = fila.archivo
    foto["FECHA_ARCHIVO"] = fila.fecha_archivo

    bitacora.append(pd.DataFrame({
        "ID_REGISTRO": foto.ID_REGISTRO.astype("int64"),
        "idx_foto": np.int16(idx),
        "DEFUNCION": pd.to_numeric(foto.DEFUNCION, errors="coerce").astype("Int8"),
    }))

    acumulado = foto if acumulado is None else pd.concat([acumulado, foto], ignore_index=True)
    acumulado = acumulado.drop_duplicates("ID_REGISTRO", keep="last")

dengue = acumulado.reset_index(drop=True)
del acumulado

print(f"Filas leídas (todas las fotos): {filas_leidas:,}")
print(f"Registros únicos tras deduplicar: {len(dengue):,}")
print(f"IDs duplicados dentro de una misma foto: {dups_internos:,}")

Filas leídas (todas las fotos): 19,876,426
Registros únicos tras deduplicar: 1,276,897
IDs duplicados dentro de una misma foto: 148


## 4. Metadatos de trazabilidad
- `N_FOTOS`: en cuántas fotos semanales apareció el registro.
- `PRIMERA_FOTO` / `ULTIMA_FOTO`: fecha de la primera y la última foto donde aparece.
- `DEFUNCION_CAMBIO`: `True` si el valor de `DEFUNCION` cambió en algún momento (por ejemplo, de No → Sí).
- `EN_FOTO_FINAL`: `True` si el registro sigue en la **última foto de su año**. Los `False` son registros que la DGE dio de baja después. Hay que decidir qué hacer con ellos durante la limpieza.

In [5]:
log = pd.concat(bitacora, ignore_index=True)
del bitacora

traza = log.groupby("ID_REGISTRO").agg(
    N_FOTOS=("idx_foto", "size"),
    idx_primera=("idx_foto", "min"),
    idx_ultima=("idx_foto", "max"),
    DEFUNCION_N_VALORES=("DEFUNCION", "nunique"),
)
traza["PRIMERA_FOTO"] = inventario.fecha_archivo.values[traza.idx_primera]
traza["ULTIMA_FOTO"] = inventario.fecha_archivo.values[traza.idx_ultima]
traza["DEFUNCION_CAMBIO"] = traza.DEFUNCION_N_VALORES > 1

# Última foto de cada año (según el nombre del archivo)
idx_finales = inventario.groupby("anio_archivo").tail(1).index
ids_finales = set(log.loc[log.idx_foto.isin(idx_finales), "ID_REGISTRO"])
traza["EN_FOTO_FINAL"] = traza.index.isin(ids_finales)
del log

traza = traza[["N_FOTOS", "PRIMERA_FOTO", "ULTIMA_FOTO", "DEFUNCION_CAMBIO", "EN_FOTO_FINAL"]]

dengue["ID_REGISTRO"] = dengue.ID_REGISTRO.astype("int64")
dengue = dengue.merge(traza, left_on="ID_REGISTRO", right_index=True, how="left", validate="one_to_one")
traza[["DEFUNCION_CAMBIO", "EN_FOTO_FINAL"]].apply(pd.Series.value_counts)

,DEFUNCION_CAMBIO,EN_FOTO_FINAL
False,1274403,22629
True,2494,1254268


## 5. Normalización de tipos
- Las fechas vienen en dos formatos (`AAAA-MM-DD` y `DD/MM/AAAA`). `9999-99-99` significa "sin registro" y se convierte en `NaT`.
- Las demás columnas son claves numéricas de catálogo. Se pasan a enteros *nullable* (`Int64`) y se conservan los códigos originales; no se recodifica nada todavía.

In [6]:
def parse_fecha(s: pd.Series) -> pd.Series:
    iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    dmy = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    return iso.fillna(dmy)

for col in ["FECHA_ACTUALIZACION", "FECHA_SIGN_SINTOMAS"]:
    raw = dengue[col]
    dengue[col] = parse_fecha(raw)
    no_parse = raw.notna() & dengue[col].isna() & ~raw.str.startswith("9999", na=False)
    print(f"{col}: {no_parse.sum()} valores no interpretables", raw[no_parse].unique()[:5])

COLS_FECHA = ["FECHA_ACTUALIZACION", "FECHA_SIGN_SINTOMAS"]
COLS_CODIGO = [c for c in COLUMNAS if c not in COLS_FECHA + ["ID_REGISTRO"]]
for col in COLS_CODIGO:
    dengue[col] = pd.to_numeric(dengue[col], errors="raise").astype("Int64")

dengue["ANIO_SINTOMAS"] = dengue.FECHA_SIGN_SINTOMAS.dt.year.astype("Int64")
dengue.dtypes

FECHA_ACTUALIZACION: 0 valores no interpretables <StringArray>
[]
Length: 0, dtype: str
FECHA_SIGN_SINTOMAS: 0 valores no interpretables <StringArray>
[]
Length: 0, dtype: str


FECHA_ACTUALIZACION       datetime64[us]
ID_REGISTRO                        int64
SEXO                               Int64
EDAD_ANOS                          Int64
ENTIDAD_RES                        Int64
MUNICIPIO_RES                      Int64
HABLA_LENGUA_INDIG                 Int64
INDIGENA                           Int64
ENTIDAD_UM_NOTIF                   Int64
MUNICIPIO_UM_NOTIF                 Int64
INSTITUCION_UM_NOTIF               Int64
FECHA_SIGN_SINTOMAS       datetime64[us]
TIPO_PACIENTE                      Int64
HEMORRAGICOS                       Int64
DIABETES                           Int64
HIPERTENSION                       Int64
ENFERMEDAD_ULC_PEPTICA             Int64
ENFERMEDAD_RENAL                   Int64
INMUNOSUPR                         Int64
CIRROSIS_HEPATICA                  Int64
EMBARAZO                           Int64
DEFUNCION                          Int64
DICTAMEN                           Int64
TOMA_MUESTRA                       Int64
RESULTADO_PCR   

## 6. Validaciones rápidas

In [7]:
assert dengue.ID_REGISTRO.is_unique
print(f"Shape final: {dengue.shape}")
print(f"Síntomas: {dengue.FECHA_SIGN_SINTOMAS.min():%d-%m-%Y} → {dengue.FECHA_SIGN_SINTOMAS.max():%d-%m-%Y}")
dengue.ANIO_SINTOMAS.value_counts().sort_index().rename("registros").to_frame()

Shape final: (1276897, 36)
Síntomas: 01-01-2020 → 31-10-2026


,registros
ANIO_SINTOMAS,
2020,120671
2021,37395
2022,61123
2023,281916
2024,555685
2025,146444
2026,73663


In [8]:
# Catálogo ESTATUS_CASO: 1 = Probable, 2 = Confirmado, 3 = Descartado | DEFUNCION: 1 = Sí, 2 = No
pd.crosstab([dengue.ANIO_SINTOMAS, dengue.ESTATUS_CASO], dengue.DEFUNCION, margins=True)

DEFUNCION                      1        2      All
ANIO_SINTOMAS ESTATUS_CASO                        
2020          1               44    67574    67618
              2              142    24172    24314
              3              179    28560    28739
2021          1               22    15715    15737
              2               50     6711     6761
              3              158    14739    14897
2022          1               48    28060    28108
              2               83    12616    12699
              3              272    20044    20316
2023          1              135   173641   173776
              2              396    54144    54540
              3              473    53127    53600
2024          1              296   338924   339220
              2              856   122481   123337
              3              872    92256    93128
2025          1               70    77219    77289
              2              188    21845    22033
              3              546    46576    47122
2026          1               45    37099    37144
              2               55     9428     9483
              3              366    26670    27036
All                         5296  1271601  1276897

In [9]:
dengue.head()

,FECHA_ACTUALIZACION,ID_REGISTRO,SEXO,EDAD_ANOS,ENTIDAD_RES,MUNICIPIO_RES,HABLA_LENGUA_INDIG,INDIGENA,ENTIDAD_UM_NOTIF,MUNICIPIO_UM_NOTIF,INSTITUCION_UM_NOTIF,FECHA_SIGN_SINTOMAS,TIPO_PACIENTE,HEMORRAGICOS,DIABETES,HIPERTENSION,ENFERMEDAD_ULC_PEPTICA,ENFERMEDAD_RENAL,INMUNOSUPR,CIRROSIS_HEPATICA,EMBARAZO,DEFUNCION,DICTAMEN,TOMA_MUESTRA,RESULTADO_PCR,ESTATUS_CASO,ENTIDAD_ASIG,MUNICIPIO_ASIG,ARCHIVO_FUENTE,FECHA_ARCHIVO,N_FOTOS,PRIMERA_FOTO,ULTIMA_FOTO,DEFUNCION_CAMBIO,EN_FOTO_FINAL,ANIO_SINTOMAS
0,2020-12-09,812143,2,51,17,30,2,2,17,6,12,2020-05-28,1,2,1,2,2,2,2,2,2,2,5,2,5,1,17,30,datos_abiertos_dengue_101220.zip,2020-12-10,1,2020-12-10,2020-12-10,False,False,2020
1,2020-12-09,863155,1,38,17,29,2,2,17,6,12,2020-10-27,1,2,2,2,2,2,2,2,2,2,5,2,5,1,17,29,datos_abiertos_dengue_101220.zip,2020-12-10,1,2020-12-10,2020-12-10,False,False,2020
2,2020-12-09,863894,1,38,17,29,2,2,17,6,12,2020-10-27,1,2,2,2,2,2,2,2,2,2,5,2,5,1,17,29,datos_abiertos_dengue_101220.zip,2020-12-10,1,2020-12-10,2020-12-10,False,False,2020
3,2020-12-09,875044,1,6,17,12,2,2,17,31,4,2020-11-17,2,2,2,2,2,2,2,2,2,2,5,2,5,1,17,12,datos_abiertos_dengue_101220.zip,2020-12-10,1,2020-12-10,2020-12-10,False,False,2020
4,2020-12-09,875491,1,32,31,18,2,2,31,56,5,2020-11-20,1,2,2,2,2,2,2,2,2,2,5,1,5,1,31,18,datos_abiertos_dengue_101220.zip,2020-12-10,1,2020-12-10,2020-12-10,False,False,2020


## 7. Guardado
- `dengue_merge.csv`: base completa sin comprimir (~220 MB, **más que el límite de 100 MB de GitHub**).
- `dengue_merge.csv.gz`: la misma base comprimida, para subirla al repo. Se lee con `pd.read_csv("...csv.gz")`.

In [10]:
cols_salida = COLUMNAS + ["ANIO_SINTOMAS", "ARCHIVO_FUENTE", "FECHA_ARCHIVO",
                          "N_FOTOS", "PRIMERA_FOTO", "ULTIMA_FOTO", "DEFUNCION_CAMBIO", "EN_FOTO_FINAL"]
dengue = dengue[cols_salida].sort_values(["FECHA_SIGN_SINTOMAS", "ID_REGISTRO"]).reset_index(drop=True)

dengue.to_csv(OUT_DIR / "dengue_merge.csv", index=False, date_format="%Y-%m-%d")
dengue.to_csv(OUT_DIR / "dengue_merge.csv.gz", index=False, date_format="%Y-%m-%d", compression="gzip")

for f in sorted(OUT_DIR.iterdir()):
    print(f"{f.name}: {f.stat().st_size / 1e6:,.1f} MB")

dengue_merge.csv: 218.3 MB
dengue_merge.csv.gz: 15.4 MB
